[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jman4162/sensortwin-transformer-agent/blob/master/notebooks/06_label_efficiency_colab.ipynb)

# SensorTwin — label efficiency: does masked pretraining help when labels are scarce?

Runs `scripts/label_efficiency_sweep` (same as `make label-efficiency-full`): masked-patch
pretraining once per seed, then five arms (scratch transformer / pretrained + fine-tune /
pretrained + linear probe / CNN / XGBoost) at 1 / 5 / 10 / 100% label fractions, scored by test
macro-F1.

**Budget parity:** both transformer arms select their learning rate from the same two-point
validation budget (1e-3, 1e-4), so a null result cannot be an artifact of the fine-tune arm
training at a fixed lower LR — the confound in the superseded 2026-06-25 run. An earlier
single-seed run under that confound showed pretraining not helping; this matched-budget,
multi-seed run is the real verdict either way.

Budget: the 100% fraction dominates — expect several hours on a T4 for 3 seeds; the per-seed loop
prints progress. Artifacts land in `reports/experiment_summaries/label_efficiency_summary.{json,md}`.

In [1]:
!nvidia-smi

Fri Jul 24 03:16:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Opened from the Colab badge? Only the notebook is present — clone the public repo, then install.
import os

if not os.path.exists("sensortwin"):
    !git clone https://github.com/jman4162/sensortwin-transformer-agent.git
    %cd sensortwin-transformer-agent
%pip install -q -e ".[ml]"

Cloning into 'sensortwin-transformer-agent'...
remote: Enumerating objects: 549, done.
remote: Counting objects: 100% (549/549), done.
remote: Compressing objects: 100% (315/315), done.
remote: Total 549 (delta 292), reused 474 (delta 219), pack-reused 0 (from 0)
Receiving objects: 100% (549/549), 1.49 MiB | 4.23 MiB/s, done.
Resolving deltas: 100% (292/292), done.
/content/sensortwin-transformer-agent
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sensortwin (pyproject.toml) ... done


In [3]:
import torch

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch 2.11.0+cu128 | CUDA: True
GPU: NVIDIA L4


In [4]:
SEEDS = 3            # integer count: seeds 0..N-1
EPOCHS_PRETRAIN = 50
EPOCHS_FINETUNE = 30
OUT = "reports/experiment_summaries"

In [5]:
from scripts.label_efficiency_sweep import main as sweep

sweep([
    "--mode", "colab_standard",
    "--seeds", str(SEEDS),
    "--epochs-pretrain", str(EPOCHS_PRETRAIN),
    "--epochs-finetune", str(EPOCHS_FINETUNE),
    "--out", OUT,
])

Generating 20000 samples (T=512, seed=0)...
Pretraining (masked reconstruction) for 50 epochs...
  pretrain val MSE: 0.9729 -> 0.9750
--- seed 0 fraction 1% ---
  scratch: macro-F1=0.118
  pretrained_ft: macro-F1=0.177
  pretrained_probe: macro-F1=0.090
  cnn: macro-F1=0.384
  xgboost: macro-F1=0.475
--- seed 0 fraction 5% ---
  scratch: macro-F1=0.327
  pretrained_ft: macro-F1=0.449
  pretrained_probe: macro-F1=0.164
  cnn: macro-F1=0.504
  xgboost: macro-F1=0.580
--- seed 0 fraction 10% ---
  scratch: macro-F1=0.715
  pretrained_ft: macro-F1=0.649
  pretrained_probe: macro-F1=0.234
  cnn: macro-F1=0.590
  xgboost: macro-F1=0.628
--- seed 0 fraction 100% ---
  scratch: macro-F1=0.870
  pretrained_ft: macro-F1=0.870
  pretrained_probe: macro-F1=0.245
  cnn: macro-F1=0.779
  xgboost: macro-F1=0.708
--- seed 1 fraction 1% ---
  scratch: macro-F1=0.100
  pretrained_ft: macro-F1=0.204
  pretrained_probe: macro-F1=0.095
  cnn: macro-F1=0.334
  xgboost: macro-F1=0.422
--- seed 1 fraction 5% 

In [6]:
from pathlib import Path

from IPython.display import Markdown, display

display(Markdown(Path(f"{OUT}/label_efficiency_summary.md").read_text()))

# Label efficiency (masked pretraining vs scratch)

Mode `colab_standard`, 3 seed(s), pretrain 50 / fine-tune 30 epochs. Test macro-F1, mean ± sample std over seeds. Both transformer arms select their learning rate from the same validation budget (0.001, 0.0001), so the pretrained-vs-scratch comparison is not confounded by a fixed fine-tune LR.

| Fraction | scratch | pretrained_ft | pretrained_probe | cnn | xgboost |
| --- | ---: | ---: | ---: | ---: | ---: |
| 1% | 0.110 ± 0.009 | 0.171 ± 0.037 | 0.108 ± 0.026 | 0.353 ± 0.027 | 0.458 ± 0.031 |
| 5% | 0.444 ± 0.103 | 0.507 ± 0.101 | 0.174 ± 0.016 | 0.486 ± 0.017 | 0.573 ± 0.007 |
| 10% | 0.697 ± 0.034 | 0.686 ± 0.032 | 0.228 ± 0.005 | 0.584 ± 0.008 | 0.614 ± 0.015 |
| 100% | 0.870 ± 0.006 | 0.866 ± 0.005 | 0.250 ± 0.015 | 0.769 ± 0.009 | 0.708 ± 0.000 |

Regenerate: `python -m scripts.label_efficiency_sweep --mode colab_standard --seeds 3 --epochs-pretrain 50 --epochs-finetune 30`.


In [7]:
try:
    from google.colab import files

    files.download(f"{OUT}/label_efficiency_summary.md")
    files.download(f"{OUT}/label_efficiency_summary.json")
except Exception as e:
    print("Not in Colab or download unavailable:", e)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Reading the result

- The summary table is mean ± sample std over seeds; per-seed values are in the JSON.
- The pretraining question is the `pretrained_ft` − `scratch` gap at 1-10% fractions. If it is
  ≈ 0 or negative under this matched budget, the honest verdict is "masked pretraining does not
  improve label efficiency on this benchmark" — commit that, don't tune around it.
- Commit `label_efficiency_summary.{json,md}` so the model card's verdict traces to this run.